# Truc quan hoa ket qua do an Big Data

Notebook nay dung de xem ket qua 2 bai toan:
- Phan khuc khach hang RFM
- Market Basket Analysis (tuy chon)

In [1]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [2]:
def tim_root_du_an() -> Path:
    # Truong hop local: dang o root hoac notebooks/
    cwd = Path.cwd().resolve()
    ung_vien = [cwd, cwd.parent]

    # Truong hop Colab: repo thuong nam trong /content/<ten-repo>
    content = Path('/content')
    if content.exists():
        for p in content.iterdir():
            if p.is_dir():
                ung_vien.append(p.resolve())

    for root in ung_vien:
        if (root / 'src').exists() and (root / 'data').exists():
            return root

    return cwd

ROOT = tim_root_du_an()
RFM_SEGMENT_PATH = ROOT / 'data/3_curated/results/rfm/customer_segments'
RFM_SUMMARY_PATH = ROOT / 'data/3_curated/results/rfm/segment_summary'
MBA_RULES_PATH = ROOT / 'data/3_curated/results/mba/association_rules'

print('ROOT:', ROOT)
print('Co RFM customer_segments:', RFM_SEGMENT_PATH.exists())
print('Co RFM segment_summary:', RFM_SUMMARY_PATH.exists())
print('Co MBA association_rules:', MBA_RULES_PATH.exists())

ROOT: /mnt/c/Users/KMN12/final-bigdata-project-nhom9
Co RFM customer_segments: True
Co RFM segment_summary: True
Co MBA association_rules: True


In [3]:
rfm_segments = None
rfm_summary = None

if RFM_SEGMENT_PATH.exists() and RFM_SUMMARY_PATH.exists():
    rfm_segments = pd.read_parquet(RFM_SEGMENT_PATH)
    rfm_summary = pd.read_parquet(RFM_SUMMARY_PATH)

    print('Da nap du lieu RFM thanh cong.')
    display(rfm_segments.head())
    display(rfm_summary.sort_values('business_score', ascending=False))
else:
    print('Chua co output RFM. Hay chay pipeline truoc:')
    print('python3 src/main_pipeline.py --project rfm --step all')

Da nap du lieu RFM thanh cong.


""


KeyError: 'business_score'

## 1) So luong khach hang theo phan khuc

In [4]:
if rfm_segments is not None and not rfm_segments.empty:
    segment_counts = (
        rfm_segments['segment_label']
        .value_counts()
        .rename_axis('segment_label')
        .reset_index(name='customers')
    )

    ax = sns.barplot(
        data=segment_counts,
        x='segment_label',
        y='customers',
        hue='segment_label',
        palette='Set2',
        legend=False
    )
    ax.set_title('So luong khach hang theo nhan phan khuc')
    ax.set_xlabel('Nhan phan khuc')
    ax.set_ylabel('So khach hang')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print('Bo qua ve bieu do phan khuc vi chua co du lieu RFM.')

Bo qua ve bieu do phan khuc vi chua co du lieu RFM.


## 2) Chi so trung binh RFM theo phan khuc

In [5]:
if rfm_summary is not None and not rfm_summary.empty:
    metrics = ['avg_recency_days', 'avg_frequency', 'avg_monetary']

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    tieu_de = ['Recency TB (ngay)', 'Frequency TB', 'Monetary TB']

    for i, metric in enumerate(metrics):
        sns.barplot(
            data=rfm_summary,
            x='segment_label',
            y=metric,
            hue='segment_label',
            palette='Set3',
            legend=False,
            ax=axes[i]
        )
        axes[i].set_title(tieu_de[i])
        axes[i].set_xlabel('Nhan phan khuc')
        axes[i].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('Bo qua ve bieu do RFM vi chua co du lieu tong hop.')

Bo qua ve bieu do RFM vi chua co du lieu tong hop.


## 3) Tuy chon: Top luat MBA (neu co)

In [6]:
if MBA_RULES_PATH.exists():
    mba_rules = pd.read_parquet(MBA_RULES_PATH)

    if mba_rules.empty:
        print('File MBA co ton tai nhung dang rong (thu giam min_support).')
    else:
        top_rules = mba_rules.sort_values('confidence', ascending=False).head(15).copy()
        top_rules['rule'] = top_rules['antecedent'].astype(str) + ' => ' + top_rules['consequent'].astype(str)

        plt.figure(figsize=(12, 7))
        sns.barplot(
            data=top_rules,
            x='confidence',
            y='rule',
            hue='rule',
            palette='viridis',
            legend=False
        )
        plt.title('Top 15 luat ket hop theo confidence')
        plt.xlabel('Confidence')
        plt.ylabel('Luat')
        plt.tight_layout()
        plt.show()
else:
    print('Chua co output MBA. Hay chay: python3 src/main_pipeline.py --project mba --step all')

File MBA co ton tai nhung dang rong (thu giam min_support).
